# IndPenSim normal-trained baseline and five diagnostic experiments

This notebook contains the original model-selection study and all five follow-up experiments. Its regressors are trained on normal batches; fault-aware evaluation does not mean the regressors were trained on faults. For the final fault-inclusive study, open the separate `main.ipynb`.

This repository copy runs in Colab or local Jupyter. Saved cell outputs are cleared; completed-run evidence is in `../results/baseline/` and `../results/experiments/`. Run in a separate environment from `main.ipynb`. See `../README.md` and `../docs/REPRODUCIBILITY.md`.


## Repository setup — separate environment for the earlier study

The scientific library versions differ from `main.ipynb`. In Colab, the following cell installs/checks this notebook's versions. Locally, first install `requirements/baseline.txt`. Support-library versions not recorded in the earlier experiment are identified in that file.


In [ ]:
# Repository setup: use this notebook's own environment
EXPECTED_VERSIONS = {'numpy': '2.1.3', 'scipy': '1.15.3', 'pandas': '2.2.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.8', 'seaborn': '0.13.2', 'joblib': '1.5.3'}
REQUIREMENTS_FILE = 'requirements/baseline.txt'
import importlib.metadata
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Use Python 3.12 for this repository's documented environment.")

try:
    import google.colab
except ImportError:
    IN_GOOGLE_COLAB = False
else:
    IN_GOOGLE_COLAB = True

# Colab may install packages. Locally, install through the requirements
# file first; change this flag only if you intentionally want installation.
INSTALL_PACKAGES = IN_GOOGLE_COLAB
module_names = {"scikit-learn": "sklearn"}
installed_versions = {}
for package in EXPECTED_VERSIONS:
    try:
        installed_versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed_versions[package] = None
mismatches = {
    package: (installed_versions[package], expected)
    for package, expected in EXPECTED_VERSIONS.items()
    if installed_versions[package] != expected
}
loaded_versions = {
    package: getattr(sys.modules.get(module_names.get(package, package)), "__version__", None)
    for package in EXPECTED_VERSIONS
}

if mismatches and INSTALL_PACKAGES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        *[f"{package}=={version}" for package, version in EXPECTED_VERSIONS.items()],
    ])
elif mismatches:
    raise RuntimeError(
        f"Install this notebook's environment first: python -m pip install -r {REQUIREMENTS_FILE}. "
        "Use a separate environment from the other notebook. Mismatches: " + str(mismatches)
    )

stale = [
    package for package, loaded in loaded_versions.items()
    if loaded is not None and loaded != EXPECTED_VERSIONS[package]
]
if stale:
    raise RuntimeError(
        "Restart the Colab session or Jupyter kernel, then run from the beginning. "
        "Older packages remain loaded: " + ", ".join(stale)
    )

for package, expected in EXPECTED_VERSIONS.items():
    actual = importlib.metadata.version(package)
    if actual != expected:
        raise RuntimeError(f"Version mismatch after setup: {package} {actual}, expected {expected}.")
    print(package, actual)
print("Environment ready. Colab:", IN_GOOGLE_COLAB)
print("Setup did not train a model. Run the following cells in order.")


## 1. Import the required libraries

Run each cell from top to bottom. If you restart Colab, use **Runtime → Run all**.


In [ ]:
import os
import json
import joblib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully.")


## 2. Optional Google Drive connection and data paths

Use your own CSV path. Drive is optional in Colab and skipped in local Jupyter. New outputs go to a fresh timestamped directory.


In [ ]:
# Optional Google Drive connection; local Jupyter skips this step.
MOUNT_GOOGLE_DRIVE = True
if IN_GOOGLE_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Google Drive mount skipped. Use a local/session data path in the next cell.")


In [ ]:
# Edit DATA_PATH and OUTPUT_ROOT to use your own data and output locations.
import os
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if IN_GOOGLE_COLAB and MOUNT_GOOGLE_DRIVE:
    DEFAULT_DATA_PATH = Path("/content/drive/MyDrive/IndPenSim_Data/100_Batches_IndPenSim_V3.csv")
    DEFAULT_OUTPUT_ROOT = Path("/content/drive/MyDrive/IndPenSim_Results") / 'baseline'
else:
    DEFAULT_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "100_Batches_IndPenSim_V3.csv"
    DEFAULT_OUTPUT_ROOT = PROJECT_ROOT / "outputs" / 'baseline'

DATA_PATH = os.environ.get("INDPENSIM_DATA_PATH", str(DEFAULT_DATA_PATH))
OUTPUT_ROOT = os.environ.get("INDPENSIM_OUTPUT_ROOT", str(DEFAULT_OUTPUT_ROOT))
data_source = Path(DATA_PATH).expanduser()
if not data_source.exists():
    raise FileNotFoundError(
        f"Cannot find the data at: {data_source}\n"
        "Set DATA_PATH to the real dataset location and rerun this cell."
    )
if not data_source.is_file() or data_source.suffix.lower() != ".csv":
    raise ValueError("The earlier notebook requires the original concatenated CSV, not a directory or ZIP.")

run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_UTC")
output_path = Path(OUTPUT_ROOT).expanduser() / run_stamp
output_path.mkdir(parents=True, exist_ok=False)
OUTPUT_DIR = str(output_path)
print("Dataset:", data_source)
print("New result directory:", OUTPUT_DIR)
print("Earlier saved results are not overwritten by this new run directory.")


## 3. Load the process variables

The original file contains approximately 2,200 Raman columns. For this basic model, we load the 39 process columns and leave Raman modelling for a later experiment. All rows from all 100 batches are still used.


In [ ]:
all_columns = pd.read_csv(
    DATA_PATH,
    nrows=0
).columns.tolist()


def is_raman_column(column_name):
    try:
        float(str(column_name).strip())
        return True
    except ValueError:
        return False


raman_columns = [
    column
    for column in all_columns
    if is_raman_column(column)
]

process_columns = [
    column
    for column in all_columns
    if not is_raman_column(column)
]

print("All columns:", len(all_columns))
print("Process columns:", len(process_columns))
print("Raman columns excluded from this basic model:", len(raman_columns))


In [ ]:
df = pd.read_csv(
    DATA_PATH,
    usecols=process_columns,
    low_memory=False
)

print("Loaded dataset shape:", df.shape)
display(df.head())


## 4. Define important columns and reconstruct the batch IDs

The supplied `Batch ID` column is not reliable in this CSV. A new batch begins whenever `Time (h)` returns to a lower value. The code therefore reconstructs batch numbers 1–100 from the time resets.


In [ ]:
TIME_COL = "Time (h)"
TARGET_COL = "Penicillin concentration(P:g/L)"
FAULT_REF_COL = "Fault reference(Fault_ref:Fault ref)"
FAULT_FLAG_COL = "Fault flag"
BATCH_COL = "Batch_ID"

required_columns = [
    TIME_COL,
    TARGET_COL,
    FAULT_REF_COL,
    FAULT_FLAG_COL
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError("Required columns are missing: " + str(missing_columns))

print("Important columns were found.")


In [ ]:
# Convert process columns to numeric values where possible.
for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Identify the first row of every new fermentation batch.
new_batch = df[TIME_COL].diff().lt(0)
new_batch.iloc[0] = True

df[BATCH_COL] = new_batch.cumsum().astype("int16")

df.sort_values(
    [BATCH_COL, TIME_COL],
    inplace=True
)

df.reset_index(drop=True, inplace=True)

number_of_batches = df[BATCH_COL].nunique()

print("Reconstructed batches:", number_of_batches)
print("Minimum batch ID:", df[BATCH_COL].min())
print("Maximum batch ID:", df[BATCH_COL].max())

if number_of_batches != 100:
    raise ValueError(
        "Expected 100 batches but reconstructed "
        + str(number_of_batches)
    )


## 5. Assign and verify the documented fault-batch group

The dataset design identifies batches 91–100. This is an evaluation label, not a learned fault detector.


In [ ]:
# Official IndPenSim design:
# batches 1-90 are normal-operation batches and 91-100 are fault batches.
# The fault-related columns are retained for diagnosis, not batch assignment.

expected_fault_batches = list(range(91, 101))

df.drop(
    columns=["_Fault_Row", "_Fault_Batch"],
    inplace=True,
    errors="ignore"
)

for column in [FAULT_REF_COL, FAULT_FLAG_COL]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df["_Fault_Batch"] = (
    df[BATCH_COL]
    .isin(expected_fault_batches)
    .astype("int8")
)


def identify_regime(batch_number):
    if 1 <= batch_number <= 30:
        return "Recipe_Normal"
    if 31 <= batch_number <= 60:
        return "Operator_Normal"
    if 61 <= batch_number <= 90:
        return "APC_Normal"
    if 91 <= batch_number <= 100:
        return "Fault"
    return "Unknown"


batch_audit = (
    df.groupby(BATCH_COL, as_index=False)
    .agg(
        Rows=(TIME_COL, "size"),
        Target_Rows=(TARGET_COL, "count"),
        Has_Fault=("_Fault_Batch", "max"),
        Fault_Reference_Min=(FAULT_REF_COL, "min"),
        Fault_Reference_Max=(FAULT_REF_COL, "max"),
        Fault_Reference_Nonzero_Rows=(
            FAULT_REF_COL,
            lambda values: int(values.fillna(0).ne(0).sum())
        ),
        Fault_Flag_Min=(FAULT_FLAG_COL, "min"),
        Fault_Flag_Max=(FAULT_FLAG_COL, "max"),
        Fault_Flag_Nonzero_Rows=(
            FAULT_FLAG_COL,
            lambda values: int(values.fillna(0).ne(0).sum())
        )
    )
)

batch_audit["Regime"] = batch_audit[BATCH_COL].apply(identify_regime)

observed_batches = sorted(batch_audit[BATCH_COL].astype(int).tolist())
assigned_fault_batches = sorted(
    batch_audit.loc[batch_audit["Has_Fault"] == 1, BATCH_COL]
    .astype(int)
    .tolist()
)
assigned_normal_batches = sorted(
    batch_audit.loc[batch_audit["Has_Fault"] == 0, BATCH_COL]
    .astype(int)
    .tolist()
)

if observed_batches != list(range(1, 101)):
    raise ValueError(
        "Reconstructed batch IDs are not exactly 1-100. Observed: "
        + str(observed_batches)
    )

if assigned_fault_batches != expected_fault_batches:
    raise ValueError(
        "Fault-batch assignment failed. Assigned: "
        + str(assigned_fault_batches)
    )

if assigned_normal_batches != list(range(1, 91)):
    raise ValueError("Normal-batch assignment failed.")

if not batch_audit["Target_Rows"].gt(0).all():
    raise ValueError("At least one batch has no target measurements.")

print("Normal batches assigned: 1 to 90")
print("Fault batches assigned:", assigned_fault_batches)
print("Fault-aware batch verification passed.")
display(batch_audit)


## 6. Create the recommended fault-aware batch split

The 90 normal batches contain three operating regimes. Each regime contributes 20 training, 5 validation and 5 normal-test batches. All 10 fault batches are reserved for a separate stress test.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)


def split_normal_regime(first_batch, last_batch):
    batch_numbers = np.arange(first_batch, last_batch + 1)
    rng.shuffle(batch_numbers)

    training = batch_numbers[:20].astype(int).tolist()
    validation = batch_numbers[20:25].astype(int).tolist()
    normal_test = batch_numbers[25:30].astype(int).tolist()

    return training, validation, normal_test


recipe_train, recipe_validation, recipe_test = split_normal_regime(1, 30)
operator_train, operator_validation, operator_test = split_normal_regime(31, 60)
apc_train, apc_validation, apc_test = split_normal_regime(61, 90)

train_batches = recipe_train + operator_train + apc_train
validation_batches = recipe_validation + operator_validation + apc_validation
normal_test_batches = recipe_test + operator_test + apc_test
fault_test_batches = list(range(91, 101))

# Compatibility name used in several later cells.
test_batches = normal_test_batches.copy()

print("Training batches:", len(train_batches), train_batches)
print("Validation batches:", len(validation_batches), validation_batches)
print("Normal test batches:", len(normal_test_batches), normal_test_batches)
print("Fault test batches:", len(fault_test_batches), fault_test_batches)


In [ ]:
train_set = set(train_batches)
validation_set = set(validation_batches)
normal_test_set = set(normal_test_batches)
fault_test_set = set(fault_test_batches)

assert len(train_set) == 60
assert len(validation_set) == 15
assert len(normal_test_set) == 15
assert len(fault_test_set) == 10

assert train_set.isdisjoint(validation_set)
assert train_set.isdisjoint(normal_test_set)
assert train_set.isdisjoint(fault_test_set)
assert validation_set.isdisjoint(normal_test_set)
assert validation_set.isdisjoint(fault_test_set)
assert normal_test_set.isdisjoint(fault_test_set)

all_split_batches = (
    train_set
    | validation_set
    | normal_test_set
    | fault_test_set
)

assert all_split_batches == set(range(1, 101))

print("All split checks passed.")
print("Every batch was assigned exactly once and there is no batch leakage.")


## 7. Assign every row to its split and save the split definition


In [ ]:
batch_to_split = {}

for batch in train_batches:
    batch_to_split[batch] = "Train"

for batch in validation_batches:
    batch_to_split[batch] = "Validation"

for batch in normal_test_batches:
    batch_to_split[batch] = "Normal_Test"

for batch in fault_test_batches:
    batch_to_split[batch] = "Fault_Test"

df["Data_Split"] = df[BATCH_COL].map(batch_to_split)

if df["Data_Split"].isna().any():
    raise ValueError("Some dataset rows were not assigned to a split.")

split_order = ["Train", "Validation", "Normal_Test", "Fault_Test"]

row_split_summary = (
    df.groupby("Data_Split")
    .agg(
        Number_of_Rows=(TIME_COL, "size"),
        Number_of_Batches=(BATCH_COL, "nunique"),
        Target_Rows=(TARGET_COL, "count")
    )
    .reindex(split_order)
)

display(row_split_summary)


In [ ]:
batch_split_table = batch_audit.copy()
batch_split_table["Data_Split"] = batch_split_table[BATCH_COL].map(batch_to_split)

regime_split_table = pd.crosstab(
    index=batch_split_table["Data_Split"],
    columns=batch_split_table["Regime"],
    margins=True
)

regime_split_table = regime_split_table.reindex(split_order + ["All"])

display(regime_split_table)

fault_aware_split = {
    "random_state": RANDOM_STATE,
    "train_batches": train_batches,
    "validation_batches": validation_batches,
    "normal_test_batches": normal_test_batches,
    "fault_test_batches": fault_test_batches
}

with open(
    os.path.join(OUTPUT_DIR, "fault_aware_batch_split.json"),
    "w"
) as file:
    json.dump(fault_aware_split, file, indent=2)

batch_split_table.to_csv(
    os.path.join(OUTPUT_DIR, "fault_aware_batch_split_table.csv"),
    index=False
)

row_split_summary.to_csv(
    os.path.join(OUTPUT_DIR, "row_split_summary.csv")
)

print("Split information saved.")


### Optional: save four separate process-data CSV files

This saves the cleaned 39-process-column dataset into four physical files. It does not save the 2,200 Raman columns.


In [ ]:
SAVE_SEPARATE_SPLIT_FILES = True

if SAVE_SEPARATE_SPLIT_FILES:
    split_data_directory = os.path.join(
        OUTPUT_DIR,
        "Fault_Aware_Dataset_Splits"
    )
    os.makedirs(split_data_directory, exist_ok=True)

    split_file_names = {
        "Train": "train_normal_60_batches.csv",
        "Validation": "validation_normal_15_batches.csv",
        "Normal_Test": "test_normal_15_batches.csv",
        "Fault_Test": "test_fault_10_batches.csv"
    }

    for split_name, file_name in split_file_names.items():
        split_data = df.loc[
            df["Data_Split"] == split_name
        ].drop(columns=["_Fault_Row", "_Fault_Batch"], errors="ignore")

        split_path = os.path.join(split_data_directory, file_name)
        split_data.to_csv(split_path, index=False)

        print(split_name, split_data.shape, "saved to", split_path)


## 8. Basic exploratory data analysis

EDA below is descriptive. Model-related decisions later use training batches only.


In [ ]:
print("Cleaned process dataset shape:", df.shape)

print("\nTarget summary:")
display(df[TARGET_COL].describe())

print("\nTwenty columns with the most missing data:")
missing_percentage = (df.isna().mean() * 100).sort_values(ascending=False)
display(missing_percentage.head(20).to_frame("Missing_Percentage"))

print("\nRows per batch:")
display(df.groupby(BATCH_COL).size().describe())


In [ ]:
plt.figure(figsize=(13, 6))

for batch in train_batches[:3]:
    batch_data = df[df[BATCH_COL] == batch]
    plt.plot(
        batch_data[TIME_COL],
        batch_data[TARGET_COL],
        label="Training batch " + str(batch)
    )

plt.title("Penicillin concentration over time in three training batches")
plt.xlabel("Time (h)")
plt.ylabel("Penicillin concentration (g/L)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## 9. Select safe online process features

Target-related, offline, fault, batch-reference and Raman variables are excluded to reduce data leakage. Fault information remains available only for the separate stress-test label.


In [ ]:
allowed_words = [
    "time",
    "aeration",
    "agitator",
    "rpm",
    "sugar feed",
    "feed rate",
    "acid flow",
    "base flow",
    "flow rate",
    "dissolved oxygen",
    "oxygen",
    "o2",
    "air",
    "temperature",
    "ph",
    "pressure",
    "volume",
    "weight",
    "water",
    "cooling",
    "heating",
    "heat",
    "carbon dioxide",
    "co2",
    "oil flow",
    "paa flow"
]

excluded_words = [
    "penicillin",
    "biomass",
    "substrate",
    "phenyl",
    "yield",
    "harvest",
    "offline",
    "final",
    "future",
    "fault",
    "raman",
    "batch",
    "reference"
]

numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

BASE_FEATURES = []

for column in numeric_columns:
    column_lower = str(column).lower()

    if column == TARGET_COL:
        continue

    if any(word in column_lower for word in excluded_words):
        continue

    if column == TIME_COL or any(
        word in column_lower
        for word in allowed_words
    ):
        BASE_FEATURES.append(column)

BASE_FEATURES = list(dict.fromkeys(BASE_FEATURES))

print("Selected base features:", len(BASE_FEATURES))
for feature in BASE_FEATURES:
    print(feature)


## 10. Create history-based features within each batch

Lag, change and rolling-average features use only current and previous process information from the same batch. The target is never used to create an input feature.


In [ ]:
df_features = df.copy()
df_features.sort_values([BATCH_COL, TIME_COL], inplace=True)
df_features.reset_index(drop=True, inplace=True)

history_keywords = [
    "dissolved oxygen",
    "sugar feed",
    "temperature",
    "ph(",
    "agitator",
    "aeration"
]

HISTORY_FEATURES = []

for keyword in history_keywords:
    matching_features = [
        feature
        for feature in BASE_FEATURES
        if keyword in str(feature).lower()
    ]

    if matching_features:
        HISTORY_FEATURES.append(matching_features[0])

HISTORY_FEATURES = list(dict.fromkeys(HISTORY_FEATURES))

print("Features receiving history variables:")
for feature in HISTORY_FEATURES:
    print(feature)


In [ ]:
ENGINEERED_FEATURES = []

for column in HISTORY_FEATURES:
    lag_name = column + "_lag1"
    difference_name = column + "_difference1"
    rolling_name = column + "_previous5_mean"

    df_features[lag_name] = (
        df_features.groupby(BATCH_COL)[column].shift(1)
    )

    df_features[difference_name] = (
        df_features.groupby(BATCH_COL)[column].diff(1)
    )

    df_features[rolling_name] = (
        df_features
        .groupby(BATCH_COL, sort=False)[column]
        .transform(
            lambda values: values
            .shift(1)
            .rolling(window=5, min_periods=1)
            .mean()
        )
    )

    ENGINEERED_FEATURES.extend([
        lag_name,
        difference_name,
        rolling_name
    ])

feed_candidates = [
    column
    for column in BASE_FEATURES
    if "sugar feed" in str(column).lower()
]

if feed_candidates:
    FEED_COL = feed_candidates[0]

    time_difference = (
        df_features.groupby(BATCH_COL)[TIME_COL]
        .diff()
        .clip(lower=0)
        .fillna(0)
    )

    feed_added = (
        df_features[FEED_COL] * time_difference
    ).fillna(0)

    cumulative_feed_name = "Cumulative_Sugar_Feed"

    df_features[cumulative_feed_name] = (
        feed_added.groupby(df_features[BATCH_COL]).cumsum()
    )

    ENGINEERED_FEATURES.append(cumulative_feed_name)

ALL_FEATURES = list(dict.fromkeys(BASE_FEATURES + ENGINEERED_FEATURES))

print("Base features:", len(BASE_FEATURES))
print("Engineered features:", len(ENGINEERED_FEATURES))
print("Total candidate features:", len(ALL_FEATURES))


## 11. Construct the four modelling datasets


In [ ]:
model_df = df_features[
    df_features[TARGET_COL].notna()
].copy()

train_mask = model_df[BATCH_COL].isin(train_batches)
validation_mask = model_df[BATCH_COL].isin(validation_batches)
normal_test_mask = model_df[BATCH_COL].isin(normal_test_batches)
fault_test_mask = model_df[BATCH_COL].isin(fault_test_batches)

for split_name, split_mask in {
    "Train": train_mask,
    "Validation": validation_mask,
    "Normal Test": normal_test_mask,
    "Fault Test": fault_test_mask
}.items():
    print(
        split_name,
        "rows:", int(split_mask.sum()),
        "batches:", model_df.loc[split_mask, BATCH_COL].nunique()
    )

assert model_df.loc[train_mask, BATCH_COL].nunique() == 60
assert model_df.loc[validation_mask, BATCH_COL].nunique() == 15
assert model_df.loc[normal_test_mask, BATCH_COL].nunique() == 15
assert model_df.loc[fault_test_mask, BATCH_COL].nunique() == 10


In [ ]:
for forbidden_column in [
    TARGET_COL,
    FAULT_REF_COL,
    FAULT_FLAG_COL,
    BATCH_COL
]:
    if forbidden_column in ALL_FEATURES:
        raise ValueError(forbidden_column + " must not be a model input.")

X_train_initial = model_df.loc[train_mask, ALL_FEATURES].copy()

# Feature removal is learned from training data only.
usable_features = []

for column in X_train_initial.columns:
    has_information = X_train_initial[column].notna().any()
    has_variation = X_train_initial[column].nunique(dropna=True) > 1

    if has_information and has_variation:
        usable_features.append(column)

removed_features = [
    column
    for column in ALL_FEATURES
    if column not in usable_features
]

ALL_FEATURES = usable_features

print("Usable model features:", len(ALL_FEATURES))
print("Removed using training data:", removed_features)


In [ ]:
def make_X(split_mask):
    return model_df.loc[
        split_mask,
        ALL_FEATURES
    ].replace([np.inf, -np.inf], np.nan).copy()


X_train = make_X(train_mask)
X_validation = make_X(validation_mask)
X_normal_test = make_X(normal_test_mask)
X_fault_test = make_X(fault_test_mask)

y_train = model_df.loc[train_mask, TARGET_COL].copy()
y_validation = model_df.loc[validation_mask, TARGET_COL].copy()
y_normal_test = model_df.loc[normal_test_mask, TARGET_COL].copy()
y_fault_test = model_df.loc[fault_test_mask, TARGET_COL].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_validation:", X_validation.shape, "y_validation:", y_validation.shape)
print("X_normal_test:", X_normal_test.shape, "y_normal_test:", y_normal_test.shape)
print("X_fault_test:", X_fault_test.shape, "y_fault_test:", y_fault_test.shape)


## 12. Training-only correlation check


In [ ]:
training_eda = model_df.loc[
    train_mask,
    ALL_FEATURES + [TARGET_COL]
].copy()

correlation_matrix = training_eda.corr(numeric_only=True)

target_correlations = (
    correlation_matrix[TARGET_COL]
    .drop(TARGET_COL)
    .abs()
    .sort_values(ascending=False)
)

print("Strongest absolute correlations in training data:")
display(target_correlations.head(15).to_frame("Absolute_Correlation"))


## 13. Define and compare basic regression models

- Dummy model: a minimum baseline.
- Linear regression: a simple linear model.
- Random forest: a nonlinear tree-based model.

The best model is selected using validation RMSE only.


In [ ]:
models = {
    "Dummy Median": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DummyRegressor(strategy="median"))
    ]),

    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=150,
            max_depth=18,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
}


def evaluate_regression_model(model, X, y, dataset_name):
    predictions = model.predict(X)

    results = {
        "Dataset": dataset_name,
        "MAE": float(mean_absolute_error(y, predictions)),
        "RMSE": float(np.sqrt(mean_squared_error(y, predictions))),
        "R2": float(r2_score(y, predictions))
    }

    return results, predictions


In [ ]:
validation_results = []
trained_models = {}

for model_name, model in models.items():
    print("Training:", model_name)

    model.fit(X_train, y_train)

    result, predictions = evaluate_regression_model(
        model,
        X_validation,
        y_validation,
        "Validation"
    )

    result["Model"] = model_name
    validation_results.append(result)
    trained_models[model_name] = model

validation_results_df = (
    pd.DataFrame(validation_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

display(validation_results_df)

BEST_MODEL_NAME = validation_results_df.loc[0, "Model"]
print("Best validation model:", BEST_MODEL_NAME)


## 14. Refit the selected model and perform two final tests

After model selection, training and validation data are combined. Neither the normal-test nor fault-test batches are used for fitting.


In [ ]:
X_train_validation = pd.concat(
    [X_train, X_validation],
    axis=0
)

y_train_validation = pd.concat(
    [y_train, y_validation],
    axis=0
)

final_model = clone(models[BEST_MODEL_NAME])
final_model.fit(X_train_validation, y_train_validation)

print("Final model fitted on 75 normal batches.")


In [ ]:
normal_test_results, normal_test_predictions = evaluate_regression_model(
    final_model,
    X_normal_test,
    y_normal_test,
    "Normal Test"
)

fault_test_results, fault_test_predictions = evaluate_regression_model(
    final_model,
    X_fault_test,
    y_fault_test,
    "Fault Stress Test"
)

final_test_results_df = pd.DataFrame([
    normal_test_results,
    fault_test_results
])

display(final_test_results_df)


### How to read the final table

- **MAE:** average absolute prediction error; lower is better.
- **RMSE:** penalises large errors more strongly; lower is better.
- **R²:** 1 is excellent, 0 means no better than predicting the mean, and a negative value means poor generalisation.
- **Normal Test:** expected performance on new normal batches.
- **Fault Stress Test:** performance when the process behaves abnormally, even though the model learned only from normal batches.

A worse fault-test result is not automatically a failed project. It provides evidence that abnormal behaviour changes the process and may justify a future fault detector or adaptive digital twin.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

test_plot_information = [
    (
        y_normal_test,
        normal_test_predictions,
        "Normal Test: Actual vs Predicted"
    ),
    (
        y_fault_test,
        fault_test_predictions,
        "Fault Stress Test: Actual vs Predicted"
    )
]

for axis, (actual, predicted, title) in zip(axes, test_plot_information):
    axis.scatter(actual, predicted, alpha=0.25, s=12)

    minimum_value = min(float(actual.min()), float(np.min(predicted)))
    maximum_value = max(float(actual.max()), float(np.max(predicted)))

    axis.plot(
        [minimum_value, maximum_value],
        [minimum_value, maximum_value],
        "r--"
    )

    axis.set_title(title)
    axis.set_xlabel("Actual penicillin concentration")
    axis.set_ylabel("Predicted penicillin concentration")
    axis.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
normal_prediction_table = model_df.loc[
    normal_test_mask,
    [BATCH_COL, TIME_COL, TARGET_COL]
].copy()

normal_prediction_table["Predicted_Target"] = normal_test_predictions

fault_prediction_table = model_df.loc[
    fault_test_mask,
    [BATCH_COL, TIME_COL, TARGET_COL, FAULT_REF_COL, FAULT_FLAG_COL]
].copy()

fault_prediction_table["Predicted_Target"] = fault_test_predictions


def calculate_batch_metrics(prediction_table):
    batch_results = []

    for batch, batch_data in prediction_table.groupby(BATCH_COL):
        actual = batch_data[TARGET_COL]
        predicted = batch_data["Predicted_Target"]

        batch_results.append({
            BATCH_COL: int(batch),
            "Rows": len(batch_data),
            "MAE": float(mean_absolute_error(actual, predicted)),
            "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
            "R2": float(r2_score(actual, predicted))
        })

    return pd.DataFrame(batch_results).sort_values(BATCH_COL)


normal_batch_metrics = calculate_batch_metrics(normal_prediction_table)
fault_batch_metrics = calculate_batch_metrics(fault_prediction_table)

print("Normal test metrics by batch:")
display(normal_batch_metrics)

print("Fault stress-test metrics by batch:")
display(fault_batch_metrics)


## 15. Inspect feature importance when available


In [ ]:
final_estimator = final_model.named_steps["model"]

if hasattr(final_estimator, "feature_importances_"):
    feature_importance = pd.DataFrame({
        "Feature": ALL_FEATURES,
        "Importance": final_estimator.feature_importances_
    }).sort_values("Importance", ascending=False)

    display(feature_importance.head(20))

    plt.figure(figsize=(10, 7))
    sns.barplot(
        data=feature_importance.head(15),
        x="Importance",
        y="Feature"
    )
    plt.title("Most important model features")
    plt.tight_layout()
    plt.show()

elif hasattr(final_estimator, "coef_"):
    coefficient_table = pd.DataFrame({
        "Feature": ALL_FEATURES,
        "Coefficient": final_estimator.coef_
    })

    coefficient_table["Absolute_Coefficient"] = (
        coefficient_table["Coefficient"].abs()
    )

    coefficient_table.sort_values(
        "Absolute_Coefficient",
        ascending=False,
        inplace=True
    )

    display(coefficient_table.head(20))

else:
    print("The selected model does not provide feature importance.")


## 16. Save the model and all final results


In [ ]:
MODEL_PATH = os.path.join(
    OUTPUT_DIR,
    "indpensim_fault_aware_basic_model.joblib"
)

joblib.dump(final_model, MODEL_PATH)

model_information = {
    "time_column": TIME_COL,
    "batch_column": BATCH_COL,
    "target_column": TARGET_COL,
    "features": ALL_FEATURES,
    "best_model": BEST_MODEL_NAME,
    "random_state": RANDOM_STATE,
    "train_batches": train_batches,
    "validation_batches": validation_batches,
    "normal_test_batches": normal_test_batches,
    "fault_test_batches": fault_test_batches,
    "normal_test_results": normal_test_results,
    "fault_test_results": fault_test_results
}

with open(
    os.path.join(OUTPUT_DIR, "model_information.json"),
    "w"
) as file:
    json.dump(model_information, file, indent=2)

validation_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "validation_results.csv"),
    index=False
)

final_test_results_df.to_csv(
    os.path.join(OUTPUT_DIR, "normal_and_fault_test_results.csv"),
    index=False
)

normal_prediction_table.to_csv(
    os.path.join(OUTPUT_DIR, "normal_test_predictions.csv"),
    index=False
)

fault_prediction_table.to_csv(
    os.path.join(OUTPUT_DIR, "fault_test_predictions.csv"),
    index=False
)

normal_batch_metrics.to_csv(
    os.path.join(OUTPUT_DIR, "normal_test_metrics_by_batch.csv"),
    index=False
)

fault_batch_metrics.to_csv(
    os.path.join(OUTPUT_DIR, "fault_test_metrics_by_batch.csv"),
    index=False
)

print("Model saved:", MODEL_PATH)
print("All results saved in:", OUTPUT_DIR)


What have i done till now?

> I reconstructed the 100 fermentation batches from time resets and checked the dataset's real fault labels. I split complete batches instead of individual rows to prevent data leakage. The normal data were balanced across recipe-driven, operator-controlled and APC regimes. Sixty normal batches were used for initial training, fifteen normal batches for model selection, and fifteen unseen normal batches for final evaluation. All ten fault batches were kept completely unseen as a separate stress test. After selecting the best model, I refitted it on the 75 training-plus-validation normal batches and reported normal and fault performance separately.


# Part II — Five additional strength experiments

Run this part only after all baseline cells above have completed. The code uses the same reconstructed batches, target, 36 leakage-controlled predictors, and fixed fault-aware split.

The experiments are:

1. Repeated regime-stratified complete-batch cross-validation.
2. Time/cumulative-feed ablation.
3. Fault-onset error analysis.
4. Stronger model comparison using HistGradientBoosting.
5. Early out-of-distribution (OOD) and uncertainty analysis.

Every experiment saves its tables and 300-dpi figures in `Five_Additional_Experiments` inside the existing results directory. Fault batches 91–100 remain completely excluded from all normal-model fitting and validation.


## 17. Shared experimental setup

This cell creates fresh model pipelines and common metric functions. `HistGradientBoosting` has internal row-wise early stopping disabled because a random row validation subset would mix observations from the same fermentation batches.


In [ ]:
import platform
import shutil
import time

import sklearn
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest

EXPERIMENT_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "Five_Additional_Experiments"
)
os.makedirs(EXPERIMENT_OUTPUT_DIR, exist_ok=True)

required_objects = [
    "model_df",
    "ALL_FEATURES",
    "train_batches",
    "validation_batches",
    "normal_test_batches",
    "fault_test_batches",
    "TIME_COL",
    "TARGET_COL",
    "BATCH_COL",
    "FAULT_REF_COL"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Run all baseline cells first. Missing objects: "
        + str(missing_objects)
    )

if set(range(91, 101)) != set(fault_test_batches):
    raise ValueError("Fault test must contain exactly batches 91-100.")

if set(train_batches) & set(validation_batches):
    raise ValueError("Training and validation batches overlap.")

sns.set_theme(style="whitegrid")


def build_research_models():
    """Return new, unfitted pipelines for fair repeated use."""
    return {
        "Dummy Median": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DummyRegressor(strategy="median"))
        ]),

        "Linear Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=150,
                max_depth=18,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=-1
            ))
        ]),

        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                learning_rate=0.05,
                max_iter=300,
                max_leaf_nodes=31,
                min_samples_leaf=20,
                l2_regularization=1.0,
                early_stopping=False,
                random_state=RANDOM_STATE
            ))
        ])
    }


def safe_r2(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    if len(actual) < 2 or np.nanstd(actual) == 0:
        return np.nan

    return float(r2_score(actual, predicted))


def metric_dictionary(actual, predicted):
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": safe_r2(actual, predicted)
    }


def batch_metric_table(batch_ids, actual, predicted):
    temporary = pd.DataFrame({
        BATCH_COL: np.asarray(batch_ids),
        "Actual": np.asarray(actual),
        "Predicted": np.asarray(predicted)
    })

    records = []

    for batch, group in temporary.groupby(BATCH_COL):
        metrics = metric_dictionary(group["Actual"], group["Predicted"])
        records.append({
            BATCH_COL: int(batch),
            "Rows": int(len(group)),
            **metrics
        })

    return pd.DataFrame(records).sort_values(BATCH_COL).reset_index(drop=True)


def higher_quantile(values, quantile):
    values = np.asarray(values, dtype=float)
    try:
        return float(np.quantile(values, quantile, method="higher"))
    except TypeError:
        return float(np.quantile(values, quantile, interpolation="higher"))


def save_figure(filename):
    path = os.path.join(EXPERIMENT_OUTPUT_DIR, filename)
    plt.savefig(path, dpi=300, bbox_inches="tight")
    print("Figure saved:", path)


print("Additional-experiment setup complete.")
print("Output directory:", EXPERIMENT_OUTPUT_DIR)


## 18. Experiment 1 — Repeated regime-stratified batch cross-validation

Five repeats of five-fold cross-validation create 25 held-out evaluations. In every fold, entire batches—not individual rows—are assigned to training or testing. Each test fold contains six Recipe, six Operator and six APC batches, so the three normal regimes remain balanced.

The default runs the selected Random Forest. To obtain repeated-CV distributions for all substantive models, change `CV_MODELS_TO_RUN` to `['Linear Regression', 'Random Forest', 'HistGradientBoosting']`. That option is substantially slower.


In [ ]:
CV_NUMBER_OF_FOLDS = 5
CV_NUMBER_OF_REPEATS = 5
CV_MODELS_TO_RUN = ["Random Forest"]

normal_regime_batches = {
    "Recipe_Normal": list(range(1, 31)),
    "Operator_Normal": list(range(31, 61)),
    "APC_Normal": list(range(61, 91))
}


def repeated_regime_stratified_batch_folds(
    number_of_repeats,
    number_of_folds,
    random_state
):
    fold_definitions = []

    for repeat in range(1, number_of_repeats + 1):
        repeat_rng = np.random.default_rng(random_state + repeat - 1)
        test_folds = {fold: [] for fold in range(number_of_folds)}

        for regime, batches in normal_regime_batches.items():
            shuffled = np.asarray(batches, dtype=int).copy()
            repeat_rng.shuffle(shuffled)
            regime_parts = np.array_split(shuffled, number_of_folds)

            for fold, part in enumerate(regime_parts):
                test_folds[fold].extend(part.astype(int).tolist())

        for fold in range(number_of_folds):
            test_batches_fold = sorted(test_folds[fold])
            train_batches_fold = sorted(
                set(range(1, 91)) - set(test_batches_fold)
            )

            test_regime_counts = {
                regime: len(set(test_batches_fold) & set(batches))
                for regime, batches in normal_regime_batches.items()
            }

            if set(train_batches_fold) & set(test_batches_fold):
                raise AssertionError("A batch appears in both CV partitions.")

            if set(train_batches_fold) | set(test_batches_fold) != set(range(1, 91)):
                raise AssertionError("A normal batch is missing from the CV fold.")

            if len(set(test_regime_counts.values())) != 1:
                raise AssertionError("CV test fold is not regime balanced.")

            fold_definitions.append({
                "Repeat": repeat,
                "Fold": fold + 1,
                "Train_Batches": train_batches_fold,
                "Test_Batches": test_batches_fold,
                "Test_Regime_Counts": test_regime_counts
            })

    return fold_definitions


cv_fold_definitions = repeated_regime_stratified_batch_folds(
    CV_NUMBER_OF_REPEATS,
    CV_NUMBER_OF_FOLDS,
    RANDOM_STATE
)

print("Total CV evaluations per model:", len(cv_fold_definitions))
print("Training batches in each fold:", len(cv_fold_definitions[0]["Train_Batches"]))
print("Test batches in each fold:", len(cv_fold_definitions[0]["Test_Batches"]))
print("Test regime counts:", cv_fold_definitions[0]["Test_Regime_Counts"])


In [ ]:
cv_records = []
cv_start_time = time.time()

available_cv_models = build_research_models()
unknown_cv_models = sorted(set(CV_MODELS_TO_RUN) - set(available_cv_models))

if unknown_cv_models:
    raise ValueError("Unknown CV models: " + str(unknown_cv_models))

for model_name in CV_MODELS_TO_RUN:
    for definition in cv_fold_definitions:
        fold_train_mask = model_df[BATCH_COL].isin(definition["Train_Batches"])
        fold_test_mask = model_df[BATCH_COL].isin(definition["Test_Batches"])

        X_fold_train = (
            model_df.loc[fold_train_mask, ALL_FEATURES]
            .replace([np.inf, -np.inf], np.nan)
            .copy()
        )
        y_fold_train = model_df.loc[fold_train_mask, TARGET_COL].copy()

        X_fold_test = (
            model_df.loc[fold_test_mask, ALL_FEATURES]
            .replace([np.inf, -np.inf], np.nan)
            .copy()
        )
        y_fold_test = model_df.loc[fold_test_mask, TARGET_COL].copy()
        fold_test_batch_ids = model_df.loc[fold_test_mask, BATCH_COL].copy()

        fold_model = build_research_models()[model_name]
        fit_start = time.time()
        fold_model.fit(X_fold_train, y_fold_train)
        fold_predictions = fold_model.predict(X_fold_test)
        fit_seconds = time.time() - fit_start

        row_metrics = metric_dictionary(y_fold_test, fold_predictions)
        per_batch = batch_metric_table(
            fold_test_batch_ids,
            y_fold_test,
            fold_predictions
        )

        cv_records.append({
            "Model": model_name,
            "Repeat": definition["Repeat"],
            "Fold": definition["Fold"],
            "Train_Batches": len(definition["Train_Batches"]),
            "Test_Batches": len(definition["Test_Batches"]),
            "Test_Rows": int(len(y_fold_test)),
            "Row_MAE": row_metrics["MAE"],
            "Row_RMSE": row_metrics["RMSE"],
            "Row_R2": row_metrics["R2"],
            "Mean_Batch_MAE": float(per_batch["MAE"].mean()),
            "Mean_Batch_RMSE": float(per_batch["RMSE"].mean()),
            "Median_Batch_R2": float(per_batch["R2"].median()),
            "Negative_Batch_R2_Count": int(per_batch["R2"].lt(0).sum()),
            "Fit_Seconds": float(fit_seconds),
            "Test_Batch_IDs": json.dumps(definition["Test_Batches"])
        })

        print(
            model_name,
            "repeat", definition["Repeat"],
            "fold", definition["Fold"],
            "RMSE", round(row_metrics["RMSE"], 4),
            "R2", round(row_metrics["R2"], 4)
        )

repeated_cv_results = pd.DataFrame(cv_records)
print("Repeated CV completed in minutes:", round((time.time() - cv_start_time) / 60, 2))
display(repeated_cv_results.head())


In [ ]:
cv_metric_columns = [
    "Row_MAE",
    "Row_RMSE",
    "Row_R2",
    "Mean_Batch_MAE",
    "Mean_Batch_RMSE",
    "Median_Batch_R2",
    "Negative_Batch_R2_Count"
]

repeated_cv_long = repeated_cv_results.melt(
    id_vars=["Model", "Repeat", "Fold"],
    value_vars=cv_metric_columns,
    var_name="Metric",
    value_name="Value"
)

repeated_cv_summary = (
    repeated_cv_long
    .groupby(["Model", "Metric"], as_index=False)
    .agg(
        Evaluations=("Value", "count"),
        Mean=("Value", "mean"),
        SD=("Value", "std"),
        Median=("Value", "median"),
        Minimum=("Value", "min"),
        Maximum=("Value", "max"),
        Empirical_2_5_Percentile=("Value", lambda x: np.quantile(x, 0.025)),
        Empirical_97_5_Percentile=("Value", lambda x: np.quantile(x, 0.975))
    )
)

repeated_cv_results.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_1_repeated_batch_cv_fold_metrics.csv"),
    index=False
)
repeated_cv_summary.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_1_repeated_batch_cv_summary.csv"),
    index=False
)

display(repeated_cv_summary)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

sns.boxplot(
    data=repeated_cv_results,
    x="Model",
    y="Row_RMSE",
    ax=axes[0],
    color="#4C78A8"
)
sns.stripplot(
    data=repeated_cv_results,
    x="Model",
    y="Row_RMSE",
    ax=axes[0],
    color="black",
    alpha=0.55,
    size=4
)
axes[0].set_title("Repeated complete-batch CV: RMSE")
axes[0].set_xlabel("")
axes[0].set_ylabel("Pooled row RMSE (g/L)")
axes[0].tick_params(axis="x", rotation=15)

sns.boxplot(
    data=repeated_cv_results,
    x="Model",
    y="Row_R2",
    ax=axes[1],
    color="#59A14F"
)
sns.stripplot(
    data=repeated_cv_results,
    x="Model",
    y="Row_R2",
    ax=axes[1],
    color="black",
    alpha=0.55,
    size=4
)
axes[1].set_title("Repeated complete-batch CV: R-squared")
axes[1].set_xlabel("")
axes[1].set_ylabel("Pooled row R-squared")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
save_figure("figure_experiment_1_cv_distribution.png")
plt.show()


**Report this experiment using the mean, standard deviation and empirical range across all 25 folds.** The 2.5th–97.5th percentiles describe the observed split-to-split distribution; they should not be presented as a formal confidence interval because repeated folds are correlated. Give both pooled row metrics and macro batch metrics so long batches do not dominate the conclusion.


## 19. Experiment 2 — Time and cumulative-feed ablation

Four Random Forests are fitted on the same 75 normal training-plus-validation batches and evaluated on the unchanged 15 normal-test and 10 fault-test batches:

- All features.
- Without raw fermentation time.
- Without cumulative sugar feed.
- Without both variables.

This directly tests whether the baseline performance depends on batch-progress shortcuts. The ablation removes the exact raw columns only; all other process and causal history features remain unchanged.


In [ ]:
CUMULATIVE_FEED_COL = "Cumulative_Sugar_Feed"

required_ablation_features = [TIME_COL, CUMULATIVE_FEED_COL]
absent_ablation_features = [
    feature for feature in required_ablation_features
    if feature not in ALL_FEATURES
]

if absent_ablation_features:
    raise ValueError(
        "Required ablation features are absent: "
        + str(absent_ablation_features)
    )

ablation_feature_sets = {
    "All features": list(ALL_FEATURES),
    "Minus Time": [f for f in ALL_FEATURES if f != TIME_COL],
    "Minus cumulative feed": [
        f for f in ALL_FEATURES if f != CUMULATIVE_FEED_COL
    ],
    "Minus Time and cumulative feed": [
        f for f in ALL_FEATURES
        if f not in {TIME_COL, CUMULATIVE_FEED_COL}
    ]
}

train_validation_mask = (
    model_df[BATCH_COL].isin(train_batches + validation_batches)
)

fixed_test_definitions = {
    "Normal Test": normal_test_batches,
    "Fault Test": fault_test_batches
}

ablation_records = []
ablation_batch_records = []

for variant, feature_list in ablation_feature_sets.items():
    ablation_model = build_research_models()["Random Forest"]

    X_ablation_train = (
        model_df.loc[train_validation_mask, feature_list]
        .replace([np.inf, -np.inf], np.nan)
        .copy()
    )
    y_ablation_train = model_df.loc[train_validation_mask, TARGET_COL].copy()

    ablation_model.fit(X_ablation_train, y_ablation_train)

    for dataset_name, dataset_batches in fixed_test_definitions.items():
        dataset_mask = model_df[BATCH_COL].isin(dataset_batches)
        X_ablation_test = (
            model_df.loc[dataset_mask, feature_list]
            .replace([np.inf, -np.inf], np.nan)
            .copy()
        )
        y_ablation_test = model_df.loc[dataset_mask, TARGET_COL].copy()
        batch_ids = model_df.loc[dataset_mask, BATCH_COL].copy()

        predictions = ablation_model.predict(X_ablation_test)
        overall = metric_dictionary(y_ablation_test, predictions)
        by_batch = batch_metric_table(batch_ids, y_ablation_test, predictions)

        ablation_records.append({
            "Variant": variant,
            "Dataset": dataset_name,
            "Number_of_Features": len(feature_list),
            **overall,
            "Mean_Batch_MAE": float(by_batch["MAE"].mean()),
            "Mean_Batch_RMSE": float(by_batch["RMSE"].mean()),
            "Median_Batch_R2": float(by_batch["R2"].median()),
            "Negative_Batch_R2_Count": int(by_batch["R2"].lt(0).sum())
        })

        by_batch.insert(0, "Dataset", dataset_name)
        by_batch.insert(0, "Variant", variant)
        ablation_batch_records.append(by_batch)

    print("Completed ablation variant:", variant)

ablation_results = pd.DataFrame(ablation_records)
ablation_by_batch = pd.concat(ablation_batch_records, ignore_index=True)

all_feature_reference = (
    ablation_results.loc[
        ablation_results["Variant"] == "All features",
        ["Dataset", "RMSE"]
    ]
    .rename(columns={"RMSE": "All_Features_RMSE"})
)

ablation_results = ablation_results.merge(
    all_feature_reference,
    on="Dataset",
    how="left"
)
ablation_results["RMSE_Percent_Change_vs_All"] = (
    100
    * (ablation_results["RMSE"] - ablation_results["All_Features_RMSE"])
    / ablation_results["All_Features_RMSE"]
)

ablation_results.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_2_time_feed_ablation.csv"),
    index=False
)
ablation_by_batch.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_2_time_feed_ablation_by_batch.csv"),
    index=False
)

display(ablation_results.sort_values(["Dataset", "RMSE"]))


In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(
    data=ablation_results,
    x="Variant",
    y="RMSE",
    hue="Dataset",
    palette=["#4C78A8", "#E45756"]
)
plt.title("Time and cumulative-feed ablation")
plt.xlabel("")
plt.ylabel("RMSE (g/L)")
plt.xticks(rotation=18, ha="right")
plt.legend(title="Evaluation dataset")
plt.tight_layout()
save_figure("figure_experiment_2_time_feed_ablation.png")
plt.show()


**Interpretation rule:** a large positive `RMSE_Percent_Change_vs_All` after removing Time and/or cumulative feed supports shortcut dependence. A small change does not prove the absence of shortcut learning because other process variables may still encode batch progress. Compare both normal and fault results; an ablated model can lose normal accuracy yet become relatively more robust to faults.


## 20. Experiment 3 — Fault onset: before, during and after

The Random Forest is trained on the same 75 normal batches. For each fault batch, the binary `Fault reference` column identifies the first and last flagged time. The three mutually exclusive phases are:

- **Before fault:** observations earlier than the first flag.
- **During fault window:** all observations from the first through the last flag, including unflagged gaps between multiple fault episodes.
- **After fault:** observations later than the final flag.

This definition is recorded explicitly because several IndPenSim fault batches contain more than one flagged episode.


In [ ]:
fault_phase_model = build_research_models()["Random Forest"]

X_phase_train = (
    model_df.loc[train_validation_mask, ALL_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)
y_phase_train = model_df.loc[train_validation_mask, TARGET_COL].copy()

fault_phase_model.fit(X_phase_train, y_phase_train)

fault_rows_mask = model_df[BATCH_COL].isin(fault_test_batches)
X_phase_fault = (
    model_df.loc[fault_rows_mask, ALL_FEATURES]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

fault_phase_table = model_df.loc[
    fault_rows_mask,
    [BATCH_COL, TIME_COL, TARGET_COL, FAULT_REF_COL]
].copy()
fault_phase_table["Predicted_Target"] = fault_phase_model.predict(X_phase_fault)
fault_phase_table["Absolute_Error"] = (
    fault_phase_table[TARGET_COL]
    - fault_phase_table["Predicted_Target"]
).abs()
fault_phase_table["Squared_Error"] = (
    fault_phase_table[TARGET_COL]
    - fault_phase_table["Predicted_Target"]
) ** 2
fault_phase_table["Fault_Reference_Active"] = (
    pd.to_numeric(fault_phase_table[FAULT_REF_COL], errors="coerce")
    .fillna(0)
    .ne(0)
)
fault_phase_table["Fault_Phase"] = pd.NA

onset_records = []

for batch, group in fault_phase_table.groupby(BATCH_COL, sort=True):
    ordered = group.sort_values(TIME_COL)
    active = ordered["Fault_Reference_Active"]

    if not active.any():
        raise ValueError(
            "No active fault reference was found for batch " + str(batch)
        )

    active_times = ordered.loc[active, TIME_COL]
    onset_time = float(active_times.min())
    final_fault_time = float(active_times.max())

    previous_active = active.shift(1, fill_value=False)
    number_of_episodes = int((active & ~previous_active).sum())

    phase = np.select(
        [
            ordered[TIME_COL] < onset_time,
            ordered[TIME_COL] <= final_fault_time
        ],
        [
            "Before fault",
            "During fault window"
        ],
        default="After fault"
    )

    fault_phase_table.loc[ordered.index, "Fault_Phase"] = phase

    onset_records.append({
        BATCH_COL: int(batch),
        "First_Fault_Time_h": onset_time,
        "Last_Fault_Time_h": final_fault_time,
        "Fault_Episodes": number_of_episodes,
        "Active_Fault_Rows": int(active.sum()),
        "Batch_Rows": int(len(ordered))
    })

fault_onset_audit = pd.DataFrame(onset_records)

phase_order = [
    "Before fault",
    "During fault window",
    "After fault"
]

fault_phase_table["Fault_Phase"] = pd.Categorical(
    fault_phase_table["Fault_Phase"],
    categories=phase_order,
    ordered=True
)

display(fault_onset_audit)
display(
    pd.crosstab(
        fault_phase_table[BATCH_COL],
        fault_phase_table["Fault_Phase"]
    ).reindex(columns=phase_order)
)


In [ ]:
fault_phase_overall_records = []
fault_phase_batch_records = []

for phase in phase_order:
    phase_data = fault_phase_table.loc[
        fault_phase_table["Fault_Phase"] == phase
    ].copy()

    overall = metric_dictionary(
        phase_data[TARGET_COL],
        phase_data["Predicted_Target"]
    )

    by_batch = batch_metric_table(
        phase_data[BATCH_COL],
        phase_data[TARGET_COL],
        phase_data["Predicted_Target"]
    )
    by_batch.insert(0, "Fault_Phase", phase)
    fault_phase_batch_records.append(by_batch)

    batch_mae_values = by_batch["MAE"].dropna().to_numpy()
    bootstrap_rng = np.random.default_rng(RANDOM_STATE)
    bootstrap_means = bootstrap_rng.choice(
        batch_mae_values,
        size=(5000, len(batch_mae_values)),
        replace=True
    ).mean(axis=1)

    fault_phase_overall_records.append({
        "Fault_Phase": phase,
        "Rows": int(len(phase_data)),
        "Batches": int(phase_data[BATCH_COL].nunique()),
        **overall,
        "Mean_Batch_MAE": float(by_batch["MAE"].mean()),
        "Mean_Batch_RMSE": float(by_batch["RMSE"].mean()),
        "Median_Batch_R2": float(by_batch["R2"].median()),
        "Mean_Batch_MAE_Bootstrap_95_Lower": float(
            np.quantile(bootstrap_means, 0.025)
        ),
        "Mean_Batch_MAE_Bootstrap_95_Upper": float(
            np.quantile(bootstrap_means, 0.975)
        )
    })

fault_phase_overall = pd.DataFrame(fault_phase_overall_records)
fault_phase_by_batch = pd.concat(fault_phase_batch_records, ignore_index=True)

fault_onset_audit.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_3_fault_onset_audit.csv"),
    index=False
)
fault_phase_overall.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_3_fault_phase_overall.csv"),
    index=False
)
fault_phase_by_batch.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_3_fault_phase_by_batch.csv"),
    index=False
)
fault_phase_table.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_3_fault_phase_predictions.csv"),
    index=False
)

display(fault_phase_overall)
display(fault_phase_by_batch)


In [ ]:
phase_positions = np.arange(len(phase_order))
figure, axis = plt.subplots(figsize=(10, 6.5))

for batch, batch_data in fault_phase_by_batch.groupby(BATCH_COL):
    batch_values = (
        batch_data.set_index("Fault_Phase")
        .reindex(phase_order)["MAE"]
        .to_numpy()
    )
    axis.plot(
        phase_positions,
        batch_values,
        color="0.65",
        alpha=0.65,
        linewidth=1
    )

summary_for_plot = (
    fault_phase_overall.set_index("Fault_Phase").reindex(phase_order)
)
mean_values = summary_for_plot["Mean_Batch_MAE"].to_numpy()
lower_errors = np.maximum(0, (
    mean_values
    - summary_for_plot["Mean_Batch_MAE_Bootstrap_95_Lower"].to_numpy()
))
upper_errors = np.maximum(0, (
    summary_for_plot["Mean_Batch_MAE_Bootstrap_95_Upper"].to_numpy()
    - mean_values
))

axis.errorbar(
    phase_positions,
    mean_values,
    yerr=np.vstack([lower_errors, upper_errors]),
    color="#D62728",
    marker="o",
    markersize=8,
    linewidth=2.5,
    capsize=5,
    label="Mean batch MAE with bootstrap 95% interval"
)

axis.set_xticks(phase_positions)
axis.set_xticklabels(phase_order)
axis.set_ylabel("Batch MAE (g/L)")
axis.set_xlabel("Fault phase")
axis.set_title("Prediction error before, during and after the fault window")
axis.legend(loc="best")
axis.grid(alpha=0.25)

plt.tight_layout()
save_figure("figure_experiment_3_fault_phase_mae.png")
plt.show()


**Report MAE and RMSE as the primary phase metrics.** Phase-specific R-squared can be unstable when the target range is narrow. The grey lines in the figure show individual batches; the red line shows the mean batch MAE and a batch-level bootstrap interval. A rise beginning during the fault window supports temporal association between fault onset and prediction failure; elevated after-fault error suggests persistent process effects.


## 21. Experiment 4 — Stronger comparator

This experiment retains the Dummy Median baseline, Linear Regression and Random Forest, and adds `HistGradientBoostingRegressor`. First, every model is fitted on the original 60 training batches and assessed on the 15 validation batches. Then the unchanged model specifications are refitted on the combined 75 normal batches and evaluated on both fixed test sets.

HistGradientBoosting is the required stronger nonlinear comparator. All models receive the same 36 predictors, including the same within-batch lag, difference and rolling process features. No test result is used for hyperparameter selection.


In [ ]:
comparison_validation_records = []
comparison_test_records = []
comparison_batch_records = []

comparison_model_names = [
    "Dummy Median",
    "Linear Regression",
    "Random Forest",
    "HistGradientBoosting"
]

for model_name in comparison_model_names:
    # Stage A: original validation comparison (60 train -> 15 validation).
    validation_model = build_research_models()[model_name]
    validation_model.fit(X_train, y_train)
    validation_predictions = validation_model.predict(X_validation)
    validation_metrics = metric_dictionary(y_validation, validation_predictions)

    validation_batch_metrics = batch_metric_table(
        model_df.loc[validation_mask, BATCH_COL],
        y_validation,
        validation_predictions
    )

    comparison_validation_records.append({
        "Model": model_name,
        "Dataset": "Validation",
        **validation_metrics,
        "Mean_Batch_MAE": float(validation_batch_metrics["MAE"].mean()),
        "Mean_Batch_RMSE": float(validation_batch_metrics["RMSE"].mean()),
        "Median_Batch_R2": float(validation_batch_metrics["R2"].median())
    })

    # Stage B: exact same specification refitted on 75 normal batches.
    final_comparison_model = build_research_models()[model_name]
    final_comparison_model.fit(X_train_validation, y_train_validation)

    for dataset_name, X_test_data, y_test_data, test_mask_data in [
        ("Normal Test", X_normal_test, y_normal_test, normal_test_mask),
        ("Fault Test", X_fault_test, y_fault_test, fault_test_mask)
    ]:
        test_predictions = final_comparison_model.predict(X_test_data)
        test_metrics = metric_dictionary(y_test_data, test_predictions)
        batch_metrics = batch_metric_table(
            model_df.loc[test_mask_data, BATCH_COL],
            y_test_data,
            test_predictions
        )

        comparison_test_records.append({
            "Model": model_name,
            "Dataset": dataset_name,
            **test_metrics,
            "Mean_Batch_MAE": float(batch_metrics["MAE"].mean()),
            "Mean_Batch_RMSE": float(batch_metrics["RMSE"].mean()),
            "Median_Batch_R2": float(batch_metrics["R2"].median()),
            "Negative_Batch_R2_Count": int(batch_metrics["R2"].lt(0).sum())
        })

        batch_metrics.insert(0, "Dataset", dataset_name)
        batch_metrics.insert(0, "Model", model_name)
        comparison_batch_records.append(batch_metrics)

    print("Completed model comparison:", model_name)

model_comparison_validation = (
    pd.DataFrame(comparison_validation_records)
    .sort_values("RMSE")
    .reset_index(drop=True)
)
model_comparison_test = pd.DataFrame(comparison_test_records)
model_comparison_by_batch = pd.concat(comparison_batch_records, ignore_index=True)

model_comparison_validation.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_4_model_comparison_validation.csv"),
    index=False
)
model_comparison_test.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_4_model_comparison_test.csv"),
    index=False
)
model_comparison_by_batch.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_4_model_comparison_by_batch.csv"),
    index=False
)

print("Validation ranking (selection evidence):")
display(model_comparison_validation)
print("Fixed final-test comparison:")
display(model_comparison_test.sort_values(["Dataset", "RMSE"]))


In [ ]:
comparison_pivot = model_comparison_test.pivot(
    index="Model",
    columns="Dataset",
    values=["MAE", "RMSE", "R2"]
)

fault_degradation_records = []

for model_name in comparison_model_names:
    normal_row = model_comparison_test.loc[
        (model_comparison_test["Model"] == model_name)
        & (model_comparison_test["Dataset"] == "Normal Test")
    ].iloc[0]
    fault_row = model_comparison_test.loc[
        (model_comparison_test["Model"] == model_name)
        & (model_comparison_test["Dataset"] == "Fault Test")
    ].iloc[0]

    fault_degradation_records.append({
        "Model": model_name,
        "Normal_RMSE": normal_row["RMSE"],
        "Fault_RMSE": fault_row["RMSE"],
        "Fault_to_Normal_RMSE_Ratio": fault_row["RMSE"] / normal_row["RMSE"],
        "Normal_R2": normal_row["R2"],
        "Fault_R2": fault_row["R2"],
        "R2_Drop_Normal_to_Fault": normal_row["R2"] - fault_row["R2"]
    })

model_fault_degradation = pd.DataFrame(fault_degradation_records)
model_fault_degradation.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_4_fault_degradation.csv"),
    index=False
)

display(model_fault_degradation.sort_values("Fault_to_Normal_RMSE_Ratio"))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(
    data=model_comparison_test,
    x="Model",
    y="RMSE",
    hue="Dataset",
    ax=axes[0],
    palette=["#4C78A8", "#E45756"]
)
axes[0].set_title("Fixed-test RMSE by model")
axes[0].set_xlabel("")
axes[0].set_ylabel("RMSE (g/L)")
axes[0].tick_params(axis="x", rotation=18)

sns.barplot(
    data=model_comparison_test,
    x="Model",
    y="R2",
    hue="Dataset",
    ax=axes[1],
    palette=["#4C78A8", "#E45756"]
)
axes[1].set_title("Fixed-test R-squared by model")
axes[1].set_xlabel("")
axes[1].set_ylabel("R-squared")
axes[1].tick_params(axis="x", rotation=18)

if axes[1].get_legend() is not None:
    axes[1].get_legend().remove()

plt.tight_layout()
save_figure("figure_experiment_4_model_comparison.png")
plt.show()


**Interpretation rule:** use the validation table for model ranking and the fixed tests for final generalisation. If Linear Regression, Random Forest and HistGradientBoosting all show a substantial fault-to-normal RMSE increase, the core conclusion is not Random-Forest-specific. If only one model deteriorates, the conclusion is more dependent on model choice.


## 22. Experiment 5 — Early OOD detection and uncertainty

This experiment asks two related questions:

1. Can an unseen batch be identified as unlike the 75 normal modelling batches using only its first 12, 24, 48 or 72 hours of process data?
2. Does a validation-calibrated Random Forest interval become wider or lose coverage on fault batches?

Early OOD detection uses batch-level summaries of safe model inputs, PCA fitted only on the 75 normal modelling batches, and an Isolation Forest fitted only on those same batches. A score above the training 95th-percentile threshold is flagged as exploratory OOD. The normal test set estimates the false-positive behaviour; batches 91–100 are used only after fitting.


In [ ]:
OOD_HORIZONS_HOURS = [12, 24, 48, 72]
OOD_TRAIN_BATCHES = sorted(train_batches + validation_batches)
OOD_NORMAL_TEST_BATCHES = sorted(normal_test_batches)
OOD_FAULT_BATCHES = sorted(fault_test_batches)
OOD_ALL_BATCHES = sorted(
    OOD_TRAIN_BATCHES + OOD_NORMAL_TEST_BATCHES + OOD_FAULT_BATCHES
)

# Time is excluded because each summary is already evaluated at a fixed horizon.
OOD_INPUT_FEATURES = [
    feature for feature in ALL_FEATURES
    if feature != TIME_COL
]


def early_batch_summary(source_data, batch_numbers, horizon_hours, features):
    records = []

    for batch in batch_numbers:
        batch_data = (
            source_data.loc[
                (source_data[BATCH_COL] == batch)
                & (source_data[TIME_COL] <= horizon_hours),
                [BATCH_COL, TIME_COL] + features
            ]
            .sort_values(TIME_COL)
            .copy()
        )

        if len(batch_data) < 2:
            raise ValueError(
                "Too few observations for batch "
                + str(batch)
                + " at horizon "
                + str(horizon_hours)
            )

        record = {
            BATCH_COL: int(batch),
            "Horizon_h": int(horizon_hours),
            "Observed_Rows": int(len(batch_data))
        }

        for feature in features:
            values = pd.to_numeric(batch_data[feature], errors="coerce")
            finite_values = values.replace([np.inf, -np.inf], np.nan)

            record[feature + "__mean"] = float(finite_values.mean())
            record[feature + "__std"] = float(finite_values.std(ddof=0))

            nonmissing = finite_values.dropna()
            record[feature + "__last"] = (
                float(nonmissing.iloc[-1]) if len(nonmissing) else np.nan
            )

        records.append(record)

    return pd.DataFrame(records)


early_ood_records = []
ood_model_audit_records = []

for horizon in OOD_HORIZONS_HOURS:
    horizon_summary = early_batch_summary(
        model_df,
        OOD_ALL_BATCHES,
        horizon,
        OOD_INPUT_FEATURES
    )

    summary_feature_columns = [
        column for column in horizon_summary.columns
        if column not in {BATCH_COL, "Horizon_h", "Observed_Rows"}
    ]

    training_summary_mask = horizon_summary[BATCH_COL].isin(OOD_TRAIN_BATCHES)

    summary_imputer = SimpleImputer(strategy="median")
    summary_scaler = StandardScaler()

    X_summary_train = summary_imputer.fit_transform(
        horizon_summary.loc[training_summary_mask, summary_feature_columns]
    )
    X_summary_train = summary_scaler.fit_transform(X_summary_train)

    pca = PCA(n_components=0.95, svd_solver="full")
    Z_summary_train = pca.fit_transform(X_summary_train)

    isolation_forest = IsolationForest(
        n_estimators=500,
        contamination="auto",
        random_state=RANDOM_STATE + horizon,
        n_jobs=-1
    )
    isolation_forest.fit(Z_summary_train)

    X_summary_all = summary_imputer.transform(
        horizon_summary[summary_feature_columns]
    )
    X_summary_all = summary_scaler.transform(X_summary_all)
    Z_summary_all = pca.transform(X_summary_all)

    # IsolationForest score_samples is larger for in-distribution data.
    # Negation makes a larger value mean more out-of-distribution.
    ood_scores = -isolation_forest.score_samples(Z_summary_all)
    training_scores = ood_scores[training_summary_mask.to_numpy()]
    ood_threshold = higher_quantile(training_scores, 0.95)

    horizon_scored = horizon_summary[[BATCH_COL, "Horizon_h", "Observed_Rows"]].copy()
    horizon_scored["OOD_Score"] = ood_scores
    horizon_scored["Training_95pct_Threshold"] = ood_threshold
    horizon_scored["OOD_Excess_Above_Threshold"] = ood_scores - ood_threshold
    horizon_scored["OOD_Flag"] = ood_scores > ood_threshold
    horizon_scored["Set"] = np.select(
        [
            horizon_scored[BATCH_COL].isin(OOD_TRAIN_BATCHES),
            horizon_scored[BATCH_COL].isin(OOD_NORMAL_TEST_BATCHES),
            horizon_scored[BATCH_COL].isin(OOD_FAULT_BATCHES)
        ],
        [
            "Normal model-development",
            "Normal test",
            "Fault test"
        ],
        default="Unknown"
    )
    horizon_scored["Regime"] = horizon_scored[BATCH_COL].apply(identify_regime)

    early_ood_records.append(horizon_scored)
    ood_model_audit_records.append({
        "Horizon_h": horizon,
        "Summary_Features_Before_PCA": len(summary_feature_columns),
        "PCA_Components": int(pca.n_components_),
        "PCA_Explained_Variance": float(pca.explained_variance_ratio_.sum()),
        "Training_95pct_Threshold": float(ood_threshold)
    })

    print(
        "OOD horizon", horizon, "h:",
        pca.n_components_, "PCA components; threshold",
        round(ood_threshold, 4)
    )

early_ood_scores = pd.concat(early_ood_records, ignore_index=True)
ood_model_audit = pd.DataFrame(ood_model_audit_records)

early_ood_scores.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_early_ood_scores.csv"),
    index=False
)
ood_model_audit.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_ood_model_audit.csv"),
    index=False
)

display(ood_model_audit)


In [ ]:
ood_evaluation_summary = (
    early_ood_scores.loc[
        early_ood_scores["Set"].isin(["Normal test", "Fault test"])
    ]
    .groupby(["Horizon_h", "Set"], as_index=False)
    .agg(
        Batches=(BATCH_COL, "nunique"),
        Flagged_Batches=("OOD_Flag", "sum"),
        OOD_Flag_Rate=("OOD_Flag", "mean"),
        Median_OOD_Score=("OOD_Score", "median"),
        Mean_OOD_Excess=("OOD_Excess_Above_Threshold", "mean")
    )
)

fault_ood_only = early_ood_scores.loc[
    early_ood_scores["Set"] == "Fault test"
].copy()

earliest_detection_records = []

for batch, group in fault_ood_only.groupby(BATCH_COL):
    flagged_horizons = group.loc[group["OOD_Flag"], "Horizon_h"]
    earliest_detection_records.append({
        BATCH_COL: int(batch),
        "Earliest_OOD_Flag_h": (
            int(flagged_horizons.min()) if len(flagged_horizons) else np.nan
        ),
        "Flagged_At_Any_Horizon": bool(len(flagged_horizons))
    })

fault_ood_earliest_detection = pd.DataFrame(earliest_detection_records)

ood_evaluation_summary.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_ood_evaluation_summary.csv"),
    index=False
)
fault_ood_earliest_detection.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_fault_ood_earliest_detection.csv"),
    index=False
)

print("OOD evaluation summary:")
display(ood_evaluation_summary)

print("Fault-batch earliest OOD flags:")
display(fault_ood_earliest_detection)

print("Requested batches 91 and 100:")
display(
    early_ood_scores.loc[
        early_ood_scores[BATCH_COL].isin([91, 100]),
        [
            BATCH_COL,
            "Horizon_h",
            "OOD_Score",
            "Training_95pct_Threshold",
            "OOD_Excess_Above_Threshold",
            "OOD_Flag"
        ]
    ].sort_values([BATCH_COL, "Horizon_h"])
)

fault_ood_heatmap = (
    fault_ood_only.pivot(
        index=BATCH_COL,
        columns="Horizon_h",
        values="OOD_Excess_Above_Threshold"
    )
    .reindex(index=fault_test_batches, columns=OOD_HORIZONS_HOURS)
)

plt.figure(figsize=(9, 6.5))
sns.heatmap(
    fault_ood_heatmap,
    cmap="RdBu_r",
    center=0,
    annot=True,
    fmt=".3f",
    cbar_kws={"label": "OOD score minus training threshold"}
)
plt.title("Early OOD evidence for fault batches")
plt.xlabel("Observed batch duration (h)")
plt.ylabel("Fault batch")
plt.tight_layout()
save_figure("figure_experiment_5_fault_ood_heatmap.png")
plt.show()


### Validation-calibrated Random Forest uncertainty

For uncertainty, the Random Forest is fitted on the original 60 training batches only. The 15 complete validation batches calibrate a 90% interval whose local width is scaled by disagreement among the 150 trees. The normal and fault test sets remain unseen during fitting and calibration.

The interval is a practical uncertainty diagnostic, not a claim of strict finite-sample conformal validity: rows within a batch are temporally dependent. Report its empirical coverage and width separately for normal and fault batches.


In [ ]:
UNCERTAINTY_ALPHA = 0.10


def random_forest_prediction_statistics(fitted_pipeline, X, chunk_size=2000):
    imputer = fitted_pipeline.named_steps["imputer"]
    forest = fitted_pipeline.named_steps["model"]

    if not hasattr(forest, "estimators_"):
        raise TypeError("The uncertainty model must be a fitted Random Forest.")

    transformed = imputer.transform(
        X.replace([np.inf, -np.inf], np.nan)
    )

    prediction_means = []
    prediction_stds = []

    for start in range(0, transformed.shape[0], chunk_size):
        stop = min(start + chunk_size, transformed.shape[0])
        chunk = transformed[start:stop]
        tree_predictions = np.vstack([
            tree.predict(chunk)
            for tree in forest.estimators_
        ])

        prediction_means.append(tree_predictions.mean(axis=0))
        prediction_stds.append(tree_predictions.std(axis=0, ddof=1))

    return (
        np.concatenate(prediction_means),
        np.concatenate(prediction_stds)
    )


uncertainty_model = build_research_models()["Random Forest"]
uncertainty_model.fit(X_train, y_train)

validation_mean, validation_tree_sd = random_forest_prediction_statistics(
    uncertainty_model,
    X_validation
)

# A small positive floor prevents zero tree disagreement from creating
# numerically extreme normalized errors. The value is learned only
# from validation-batch tree spreads, with a 0.05 g/L lower bound.
uncertainty_sd_floor = max(
    0.10 * float(np.median(validation_tree_sd)),
    0.05
)

normalized_validation_errors = (
    np.abs(y_validation.to_numpy() - validation_mean)
    / (validation_tree_sd + uncertainty_sd_floor)
)

calibration_count = len(normalized_validation_errors)
conformal_quantile_level = min(
    1.0,
    np.ceil((calibration_count + 1) * (1 - UNCERTAINTY_ALPHA))
    / calibration_count
)
uncertainty_multiplier = higher_quantile(
    normalized_validation_errors,
    conformal_quantile_level
)

print("Calibration rows:", calibration_count)
print("Target interval coverage:", 1 - UNCERTAINTY_ALPHA)
print("Calibration quantile level:", conformal_quantile_level)
print("Tree-SD floor:", uncertainty_sd_floor)
print("Calibrated uncertainty multiplier:", uncertainty_multiplier)

uncertainty_row_tables = []
uncertainty_summary_records = []
uncertainty_batch_records = []

for dataset_name, X_data, y_data, mask_data in [
    ("Normal Test", X_normal_test, y_normal_test, normal_test_mask),
    ("Fault Test", X_fault_test, y_fault_test, fault_test_mask)
]:
    prediction_mean, prediction_tree_sd = random_forest_prediction_statistics(
        uncertainty_model,
        X_data
    )

    interval_half_width = uncertainty_multiplier * (
        prediction_tree_sd + uncertainty_sd_floor
    )
    lower_bound = prediction_mean - interval_half_width
    upper_bound = prediction_mean + interval_half_width
    actual_values = y_data.to_numpy()
    covered = (
        (actual_values >= lower_bound)
        & (actual_values <= upper_bound)
    )

    row_table = model_df.loc[
        mask_data,
        [BATCH_COL, TIME_COL, TARGET_COL]
    ].copy()
    row_table["Dataset"] = dataset_name
    row_table["Predicted_Target"] = prediction_mean
    row_table["Tree_SD"] = prediction_tree_sd
    row_table["Interval_Lower"] = lower_bound
    row_table["Interval_Upper"] = upper_bound
    row_table["Interval_Width"] = 2 * interval_half_width
    row_table["Covered"] = covered
    row_table["Absolute_Error"] = np.abs(actual_values - prediction_mean)
    uncertainty_row_tables.append(row_table)

    overall = metric_dictionary(actual_values, prediction_mean)
    uncertainty_summary_records.append({
        "Dataset": dataset_name,
        **overall,
        "Target_Coverage": 1 - UNCERTAINTY_ALPHA,
        "Empirical_Coverage": float(covered.mean()),
        "Mean_Interval_Width": float((2 * interval_half_width).mean()),
        "Median_Interval_Width": float(np.median(2 * interval_half_width)),
        "Mean_Tree_SD": float(prediction_tree_sd.mean())
    })

    for batch, batch_data in row_table.groupby(BATCH_COL):
        batch_metrics = metric_dictionary(
            batch_data[TARGET_COL],
            batch_data["Predicted_Target"]
        )
        uncertainty_batch_records.append({
            BATCH_COL: int(batch),
            "Dataset": dataset_name,
            **batch_metrics,
            "Empirical_Coverage": float(batch_data["Covered"].mean()),
            "Mean_Interval_Width": float(batch_data["Interval_Width"].mean()),
            "Mean_Tree_SD": float(batch_data["Tree_SD"].mean()),
            "Mean_Absolute_Error": float(batch_data["Absolute_Error"].mean())
        })

uncertainty_rows = pd.concat(uncertainty_row_tables, ignore_index=True)
uncertainty_summary = pd.DataFrame(uncertainty_summary_records)
uncertainty_by_batch = pd.DataFrame(uncertainty_batch_records)

uncertainty_summary.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_uncertainty_summary.csv"),
    index=False
)
uncertainty_by_batch.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_uncertainty_by_batch.csv"),
    index=False
)
uncertainty_rows.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_uncertainty_predictions.csv"),
    index=False
)

display(uncertainty_summary)
display(uncertainty_by_batch.sort_values("RMSE", ascending=False))


In [ ]:
OOD_LINK_HORIZON_H = 24

ood_at_link_horizon = early_ood_scores.loc[
    early_ood_scores["Horizon_h"] == OOD_LINK_HORIZON_H,
    [
        BATCH_COL,
        "Set",
        "OOD_Score",
        "Training_95pct_Threshold",
        "OOD_Excess_Above_Threshold",
        "OOD_Flag"
    ]
].copy()

ood_uncertainty_by_batch = uncertainty_by_batch.merge(
    ood_at_link_horizon,
    on=BATCH_COL,
    how="left",
    suffixes=("_Uncertainty", "_OOD")
)

ood_error_rank_correlation = float(
    ood_uncertainty_by_batch["OOD_Score"].rank()
    .corr(ood_uncertainty_by_batch["RMSE"].rank())
)
uncertainty_error_rank_correlation = float(
    ood_uncertainty_by_batch["Mean_Tree_SD"].rank()
    .corr(ood_uncertainty_by_batch["RMSE"].rank())
)

ood_uncertainty_by_batch.to_csv(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_5_ood_uncertainty_by_batch.csv"),
    index=False
)

print(
    "Spearman rank correlation: 24-h OOD score vs batch RMSE =",
    round(ood_error_rank_correlation, 4)
)
print(
    "Spearman rank correlation: mean tree SD vs batch RMSE =",
    round(uncertainty_error_rank_correlation, 4)
)

print("OOD/uncertainty results for batches 91 and 100:")
display(
    ood_uncertainty_by_batch.loc[
        ood_uncertainty_by_batch[BATCH_COL].isin([91, 100])
    ]
)

plt.figure(figsize=(10, 6.5))

palette = {
    "Normal Test": "#4C78A8",
    "Fault Test": "#E45756"
}

for dataset_name, group in ood_uncertainty_by_batch.groupby("Dataset"):
    plt.scatter(
        group["OOD_Score"],
        group["RMSE"],
        s=70,
        alpha=0.8,
        label=dataset_name,
        color=palette.get(dataset_name, "grey")
    )

threshold_for_plot = float(
    ood_at_link_horizon["Training_95pct_Threshold"].iloc[0]
)
plt.axvline(
    threshold_for_plot,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Training 95th-percentile OOD threshold"
)

for batch in [91, 100]:
    row = ood_uncertainty_by_batch.loc[
        ood_uncertainty_by_batch[BATCH_COL] == batch
    ]
    if len(row):
        plt.annotate(
            "Batch " + str(batch),
            (float(row["OOD_Score"].iloc[0]), float(row["RMSE"].iloc[0])),
            xytext=(6, 6),
            textcoords="offset points"
        )

plt.title("Early OOD score versus full-batch prediction error")
plt.xlabel("OOD score using first 24 h")
plt.ylabel("Full-batch RMSE (g/L)")
plt.legend()
plt.tight_layout()
save_figure("figure_experiment_5_ood_vs_batch_rmse.png")
plt.show()


**Interpretation rule:** an OOD flag is a warning, not proof of a fault. Check the normal-test false-positive rate and the fault-test detection rate at each horizon. For batches 91 and 100, report whether the score crosses the threshold and how early. Useful uncertainty should increase with batch error and should retain reasonable coverage; poor fault coverage despite narrow intervals means the model is overconfident outside its training distribution.


## 23. Export a report-ready result package

This final cell saves the experimental configuration, creates a concise text summary from the computed values, and packages all CSV tables and figures into one ZIP file. Send that ZIP file with the executed notebook when updating the Word report; exact results should not be written into the dissertation before the experiment has run successfully.


In [ ]:
experiment_configuration = {
    "random_state": RANDOM_STATE,
    "target_column": TARGET_COL,
    "batch_column": BATCH_COL,
    "time_column": TIME_COL,
    "fault_reference_column": FAULT_REF_COL,
    "number_of_model_features": len(ALL_FEATURES),
    "model_features": list(ALL_FEATURES),
    "normal_training_batches": list(train_batches),
    "normal_validation_batches": list(validation_batches),
    "normal_test_batches": list(normal_test_batches),
    "fault_test_batches": list(fault_test_batches),
    "repeated_cv_folds": CV_NUMBER_OF_FOLDS,
    "repeated_cv_repeats": CV_NUMBER_OF_REPEATS,
    "repeated_cv_models": CV_MODELS_TO_RUN,
    "ood_horizons_hours": OOD_HORIZONS_HOURS,
    "uncertainty_target_coverage": 1 - UNCERTAINTY_ALPHA,
    "software_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__
    },
    "fault_phase_definition": {
        "Before fault": "time earlier than first non-zero fault reference",
        "During fault window": "time from first through last non-zero fault reference, including gaps",
        "After fault": "time later than last non-zero fault reference"
    }
}

with open(
    os.path.join(EXPERIMENT_OUTPUT_DIR, "experiment_configuration.json"),
    "w",
    encoding="utf-8"
) as file:
    json.dump(experiment_configuration, file, indent=2)

cv_primary = repeated_cv_results.loc[
    repeated_cv_results["Model"] == CV_MODELS_TO_RUN[0]
]
best_validation_comparator = model_comparison_validation.iloc[0]

summary_lines = [
    "INDPENSIM FIVE ADDITIONAL EXPERIMENTS — AUTOMATIC RESULT SUMMARY",
    "",
    "Experiment 1 — Repeated complete-batch cross-validation",
    (
        f"{CV_MODELS_TO_RUN[0]} across {len(cv_primary)} evaluations: "
        f"RMSE mean {cv_primary['Row_RMSE'].mean():.4f} g/L "
        f"(SD {cv_primary['Row_RMSE'].std():.4f}); "
        f"R2 mean {cv_primary['Row_R2'].mean():.4f} "
        f"(SD {cv_primary['Row_R2'].std():.4f})."
    ),
    "",
    "Experiment 2 — Time/cumulative-feed ablation",
    ablation_results.to_string(index=False),
    "",
    "Experiment 3 — Fault phase analysis",
    fault_phase_overall.to_string(index=False),
    "",
    "Experiment 4 — Stronger comparator",
    (
        f"Best validation model by RMSE: {best_validation_comparator['Model']} "
        f"(RMSE {best_validation_comparator['RMSE']:.4f} g/L; "
        f"R2 {best_validation_comparator['R2']:.4f})."
    ),
    model_fault_degradation.to_string(index=False),
    "",
    "Experiment 5 — OOD and uncertainty",
    ood_evaluation_summary.to_string(index=False),
    "",
    uncertainty_summary.to_string(index=False),
    "",
    (
        "Spearman rank correlation, 24-h OOD score vs batch RMSE: "
        f"{ood_error_rank_correlation:.4f}"
    ),
    (
        "Spearman rank correlation, tree SD vs batch RMSE: "
        f"{uncertainty_error_rank_correlation:.4f}"
    )
]

summary_path = os.path.join(
    EXPERIMENT_OUTPUT_DIR,
    "report_ready_results_summary.txt"
)
with open(summary_path, "w", encoding="utf-8") as file:
    file.write("\n".join(summary_lines))

archive_base = os.path.join(
    OUTPUT_DIR,
    "IndPenSim_Five_Additional_Experiments_Results"
)
archive_path = shutil.make_archive(
    archive_base,
    "zip",
    EXPERIMENT_OUTPUT_DIR
)

print("All additional experiments completed.")
print("Report-ready summary:", summary_path)
print("ZIP package:", archive_path)
